# Step 5 — Load Everything into Neo4j (the "Load" stage)

This is where the previous four steps come together. For each document we now:

```
RawDocument         -> (:Document)
Chunk + embedding   -> (:Chunk)      with (:Document)-[:HAS_CHUNK]->(:Chunk)
extracted entities  -> (:Entity)     with (:Chunk)-[:MENTIONS]->(:Entity)
extracted relations -> (:Entity)-[:RELATED_TO {type}]->(:Entity)
```

The result is a **hybrid knowledge graph**: a graph of connected facts *and* a vector store of chunk embeddings, in one database. Later steps search it.

The single most important property here is **idempotency** — running the loader repeatedly must never create duplicates. We get that from `MERGE`.

> Code lives in `signalpulse/loader.py`.

## 1. Concepts

**`MERGE` vs `CREATE`.** `CREATE` always makes a new node — run it twice and you get two copies. `MERGE` means *"match if it already exists, otherwise create."* Because our schema (Step 1) puts **uniqueness constraints** on `Document.id`, `Chunk.id`, and `Entity.name`, `MERGE` on those keys is both correct and fast (the constraint's backing index makes the lookup instant).

**Upsert.** Every write is a `MERGE` followed by `SET` — create the node if new, otherwise update its properties. This "update-or-insert" is called an *upsert*, and it's what makes the whole pipeline safe to re-run on a schedule.

**The graph we build:**

| Element | Key | Notes |
|---|---|---|
| `(:Document)` | `id` (= source id) | title, url, agency, domain, `content_hash`, ... |
| `(:Chunk)` | `id` (`<doc>::<n>`) | `text` + 384-dim `embedding` (the vector store) |
| `(:Entity)` | `name` | `type` from the controlled vocabulary |
| `(:Document)-[:HAS_CHUNK]->(:Chunk)` | | document ↔ its passages |
| `(:Chunk)-[:MENTIONS]->(:Entity)` | | **provenance**: which passage stated a fact |
| `(:Entity)-[:RELATED_TO {type}]->(:Entity)` | | the knowledge triples (`AFFECTS`, ...) |

**Why link chunks to entities (`MENTIONS`)?** It gives every entity a trail back to the exact chunk — and therefore the exact source document and URL — it came from. That's what lets the chatbot answer with **citations** instead of hallucinations.

In [1]:
import os
import sys
from pathlib import Path

os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from signalpulse import connectors as C
from signalpulse import processing as P
from signalpulse import extraction as X
from signalpulse import loader as L
from signalpulse import graph

# Make sure the DB is up and the schema (constraints + indexes) exists.
graph.verify_connectivity()
graph.create_schema()   # idempotent

# Start from a clean graph so this notebook's counts are easy to read.
graph.clear_data()
print("Neo4j ready. Starting counts:", graph.node_counts())

Neo4j ready. Starting counts: {'Document': 0, 'Chunk': 0, 'Entity': 0}


## 2. Run the full pipeline for a few documents

We chain everything: **fetch (Step 2) → clean/chunk/embed (Step 3) → extract (Step 4) → load (Step 5)**. To stay within free-tier LLM limits, we cap how many chunks per document we extract from. `loader.load_document()` does all the `MERGE` writes for one document in one call.

In [2]:
MAX_CHUNKS_PER_DOC = 2   # keep LLM usage small (free-tier friendly)

# Fetch a few real documents from two sources.
docs = C.CISAKEVConnector().fetch(limit=2)
docs += C.FederalRegisterConnector(
    "centers-for-medicare-medicaid-services", "CMS", "Health IT & Civilian"
).fetch(limit=1)

# Run fetch -> chunk/embed -> extract ONCE and cache the results per document.
# (Caching lets the idempotency test below re-load without spending more LLM calls.)
prepared = []   # list of (doc, chunks, extractions)
for doc in docs:
    chunks = P.process_document(doc)[:MAX_CHUNKS_PER_DOC]        # clean + chunk + embed
    exts = [X.extract_from_chunk(c, title=doc.title) for c in chunks]  # extract (LLM)
    prepared.append((doc, chunks, exts))

# Load each prepared document into Neo4j (idempotent MERGE writes).
for doc, chunks, exts in prepared:
    stats = L.load_document(doc, chunks, exts)
    print(f"{doc.source_id:16s} | {doc.agency:5s} | "
          f"chunks={stats['chunks']} mentions={stats['entity_mentions']} "
          f"rels={stats['relationships']}")

print("\nLoad complete.")

C:\Users\saihaj\Documents\22nd Project\.venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


C:\Users\saihaj\Documents\22nd Project\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:49: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  from google.generativeai.caching import CachedContent  # type: ignore[import]


CVE-2026-48908   | CISA  | chunks=2 mentions=7 rels=3
CVE-2026-55255   | CISA  | chunks=2 mentions=6 rels=2


2026-13918       | Health and Human Services Department | chunks=1 mentions=5 rels=2

Load complete.


## 3. What's in the graph now?

`graph_summary()` counts nodes per label and relationships per type.

In [3]:
summary = L.graph_summary()
for k, v in summary.items():
    print(f"  {k:18s}: {v}")

  Document nodes    : 3
  Chunk nodes       : 5
  Entity nodes      : 13
  HAS_CHUNK rels    : 5
  MENTIONS rels     : 17
  RELATED_TO rels   : 6


## 4. Idempotency: load the same documents again

If `MERGE` is doing its job, re-loading identical data must leave every count **unchanged** (nodes/relationships updated in place, nothing duplicated).

In [4]:
before = L.graph_summary()

# Re-load the exact same prepared data (no new LLM calls — reuse the cache).
for doc, chunks, exts in prepared:
    L.load_document(doc, chunks, exts)

after = L.graph_summary()
print("before:", before)
print("after :", after)
print("\nCounts unchanged after re-load:", before == after, "  <- idempotent")

before: {'Document nodes': 3, 'Chunk nodes': 5, 'Entity nodes': 13, 'HAS_CHUNK rels': 5, 'MENTIONS rels': 17, 'RELATED_TO rels': 6}
after : {'Document nodes': 3, 'Chunk nodes': 5, 'Entity nodes': 13, 'HAS_CHUNK rels': 5, 'MENTIONS rels': 17, 'RELATED_TO rels': 6}

Counts unchanged after re-load: True   <- idempotent


## 5. Sample queries (Cypher)

A few queries that show the graph is genuinely connected and useful. Cypher reads like ASCII-art: `(node)-[:REL]->(node)`.

In [5]:
# Query 1 — each document and how many chunks it was split into
print("== Documents and their chunk counts ==")
rows = graph.run_query("""
    MATCH (d:Document)-[:HAS_CHUNK]->(c:Chunk)
    RETURN d.agency AS agency, d.title AS title, count(c) AS chunks
    ORDER BY chunks DESC
""")
for r in rows:
    print(f"  [{r['agency']}] {r['title'][:60]:60s}  chunks={r['chunks']}")

# Query 2 — knowledge triples (the extracted relationships)
print("\n== Knowledge triples (Entity)-[:RELATED_TO]->(Entity) ==")
rows = graph.run_query("""
    MATCH (a:Entity)-[r:RELATED_TO]->(b:Entity)
    RETURN a.name AS source, r.type AS rel, b.name AS target
    LIMIT 12
""")
for r in rows:
    print(f"  ({r['source']}) -{r['rel']}-> ({r['target']})")

== Documents and their chunk counts ==


  [CISA] CVE-2026-48908: JoomShaper SP Page Builder Unrestricted Uplo  chunks=2
  [CISA] CVE-2026-55255: Langflow Authorization Bypass Through User-C  chunks=2
  [Health and Human Services Department] Medicare and Medicaid Programs; Application From The Joint C  chunks=1

== Knowledge triples (Entity)-[:RELATED_TO]->(Entity) ==
  (JoomShaper) -ISSUED_BY-> (SP Page Builder)
  (CVE-2026-48908) -AFFECTS-> (SP Page Builder)
  (CISA) -ISSUED_BY-> (BOD 26-04)
  (CVE-2026-55255) -AFFECTS-> (Langflow)
  (The Joint Commission) -APPLIES_TO-> (Home Health Agency (HHA) Accreditation Program)
  (CMS) -ISSUED_BY-> (The Joint Commission)


In [6]:
# Query 3 — PROVENANCE: pick an entity and trace it back to its source document.
# This full-path traversal is exactly how the chatbot will build citations.
print("== Provenance: entity -> chunk -> source document (with URL) ==")
rows = graph.run_query("""
    MATCH (e:Entity)<-[:MENTIONS]-(c:Chunk)<-[:HAS_CHUNK]-(d:Document)
    RETURN e.name AS entity, e.type AS type, d.title AS document, d.url AS url
    ORDER BY entity
    LIMIT 8
""")
for r in rows:
    print(f"  {r['entity']} [{r['type']}]")
    print(f"      from: {r['document'][:55]}")
    print(f"      cite: {r['url']}")

# Query 4 — most-mentioned entities across the corpus
print("\n== Most-mentioned entities ==")
rows = graph.run_query("""
    MATCH (c:Chunk)-[:MENTIONS]->(e:Entity)
    RETURN e.name AS entity, e.type AS type, count(c) AS mentions
    ORDER BY mentions DESC, entity
    LIMIT 8
""")
for r in rows:
    print(f"  {r['mentions']}x  {r['entity']} [{r['type']}]")

== Provenance: entity -> chunk -> source document (with URL) ==


  BOD 26-04 [Regulation]
      from: CVE-2026-48908: JoomShaper SP Page Builder Unrestricted
      cite: https://nvd.nist.gov/vuln/detail/CVE-2026-48908
  BOD 26-04 [Regulation]
      from: CVE-2026-55255: Langflow Authorization Bypass Through U
      cite: https://nvd.nist.gov/vuln/detail/CVE-2026-55255
  BOD 26-04 [Regulation]
      from: CVE-2026-55255: Langflow Authorization Bypass Through U
      cite: https://nvd.nist.gov/vuln/detail/CVE-2026-55255
  BOD 26-04 [Regulation]
      from: CVE-2026-48908: JoomShaper SP Page Builder Unrestricted
      cite: https://nvd.nist.gov/vuln/detail/CVE-2026-48908
  CISA [Organization]
      from: CVE-2026-55255: Langflow Authorization Bypass Through U
      cite: https://nvd.nist.gov/vuln/detail/CVE-2026-55255
  CISA [Organization]
      from: CVE-2026-48908: JoomShaper SP Page Builder Unrestricted
      cite: https://nvd.nist.gov/vuln/detail/CVE-2026-48908
  CMS [Organization]
      from: Medicare and Medicaid Programs; Application From The Jo

## 6. See it in the Neo4j Browser

Open **http://localhost:7474**, log in, and try these in the query bar:

```cypher
// The whole graph (small enough to visualize)
MATCH (n) RETURN n LIMIT 100

// A document, its chunks, and the entities those chunks mention
MATCH p = (d:Document)-[:HAS_CHUNK]->(:Chunk)-[:MENTIONS]->(:Entity)
RETURN p LIMIT 50

// Just the knowledge triples between entities
MATCH p = (:Entity)-[:RELATED_TO]->(:Entity)
RETURN p LIMIT 50
```

You'll see nodes colored by label and can drag them around to explore.

## Recap & what's next

We built the **Load** stage:

- **Upsert with `MERGE`** on the constrained keys → the pipeline is **idempotent** (re-loading changed nothing, proven above).
- **`(:Document)-[:HAS_CHUNK]->(:Chunk)`** stores passages *and* their embeddings (our vector store).
- **`(:Chunk)-[:MENTIONS]->(:Entity)`** preserves **provenance** for citations.
- **`(:Entity)-[:RELATED_TO {type}]->(:Entity)`** stores the knowledge triples.

The graph is now a queryable, hybrid knowledge base. Everything from Step 2–4 has a home in Neo4j.

**Next — Step 6:** build the **retrieval** layer — vector (semantic) search over chunk embeddings, keyword (full-text) search, and a graph lookup — the tools the agent will use to answer questions with citations.

> Prompt to continue:
> *"Step 6: Create notebook `06_retrieval.ipynb`. Implement the retrieval tools over Neo4j: vector search on chunk embeddings, full-text keyword search, and an entity/graph lookup. Each result should carry its source document and URL for citations. Show example queries and their retrieved, cited passages."*